# Практика · Метрики й оцінка

> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.md](homework.md) ·
> Тест: [quiz.html](quiz.html)

Тут рахуються **всі** числа, які називає лекція, у тому самому порядку.

Що зробимо:

1. Візьмемо ту саму задачу, що й [тема 06](../06-text-classification/lecture.html):
   мітка «помилка чи ні» з англійського оригіналу, ознаки — з українського перекладу.
2. Дістанемо з передбачень **матрицю плутанини** — чотири числа — і виведемо
   з них руками precision, recall, F1 і частку правильних. Звіримо з `sklearn`.
3. Побудуємо модель, яка не вміє нічого, і подивимось, скільки в неї «точності».
4. **Доведемо `assert`-ом, що мікро-F1 дорівнює частці правильних рівно** — і в
   двокласовій задачі, і в задачі на 263 класи. І покажемо, де ця рівність ламається.
5. Побудуємо багатокласову задачу з природним довгим хвостом і подивимось, як
   макро й мікро розходяться тим сильніше, чим довший хвіст.
6. Покрутимо поріг і побачимо, що «найкращий поріг» залежить від ціни помилки.
7. Проженемо чотири моделі через три зерна й з'ясуємо, яка різниця є різницею.

> ⏱ Заміряно: близько **40 секунд процесорного часу** на чотирьох ядрах без
> відеокарти. Стінний час на завантаженій машині буде вдвічі-втричі більший — це
> черга, а не робота. Найдовше йдуть тридцять багатокласових прогонів (десять
> розмірів задачі × три зерна) і сім навчань логістичної регресії.
>
> Перша клітинка фіксує кількість потоків **до** імпорту `numpy`. Без цього
> навчання логістичної регресії «триває» кілька секунд замість 0.22 — потоки
> OpenMP більше чекають одне одного, ніж рахують.


## 1 · Середовище

Перша клітинка друкує версії. Числа теми рахувались саме на них; якщо в тебе інші —
краще знати про це одразу, а не наприкінці.


In [ ]:
# Потоки фіксуємо ДО імпорту numpy. Без цього OpenMP запускає стільки потоків,
# скільки ядер, і на спільній машині вони більше чекають одне одного, ніж рахують:
# те саме навчання спалює вдесятеро більше процесорного часу.
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import sys, re, glob, gettext, time, math, collections
import numpy as np
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, precision_recall_fscore_support,
                             accuracy_score, f1_score, classification_report)

NOTEBOOK_START = time.time()
NOTEBOOK_CPU = time.process_time()

print("Python  ", sys.version.split()[0])
print("numpy   ", np.__version__)
print("sklearn ", sklearn.__version__)


## 2 · Корпус: українські переклади інтерфейсів

Той самий корпус, що в усьому курсі. Кожна програма в Linux має файл перекладу `.mo`,
де лежать пари «англійський оригінал → український переклад». Ми беремо звідти
трійки: назва програми, оригінал, переклад.

Якщо української локалі на машині немає, вмикається вбудований запасний корпус.
Він маленький, тож числа вийдуть інші — зошит про це скаже прямо.


In [ ]:
def load_system_corpus():
    """Читаємо всі .mo-файли української локалі.
    Повертаємо трійки (програма, англійський оригінал, український переклад)."""
    docs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                    # зламаний або чужий формат — просто пропускаємо
        program = path.split('/')[-1][:-3]
        for src, dst in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і текстом не є
            if isinstance(src, str) and isinstance(dst, str) \
               and len(dst) > 30 and 'Project-Id' not in dst:
                docs.append((program, src, dst))
    return docs

print("шукаю файли в /usr/share/locale/uk/LC_MESSAGES/")
print("знайдено .mo-файлів:", len(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')))


In [ ]:
FALLBACK = [
("rpm", "failed to rebuild database: original database remain", "не вдалося перебудувати базу даних: початкова база даних зал"),
("virt-manager", "Error stopping pool '%s'", "Помилка під час завершення роботи буфера «%s»"),
("appstream", "Metainfo files may only contain icons of type `stock", "У файлах metainfo можуть міститися лише піктограми типів «st"),
("grub", "efibootmgr failed to register the boot entry: %s", "efibootmgr не вдалося зареєструвати запис завантаження: %s"),
("gold", "Do not warn if the stack is executable", "Не попереджати, якщо стек є виконуваним"),
("libvirt", "unexpected migration schema: %1$d", "неочікувана схема перенесення: %1$d"),
("gettext-tools", "%s:%d: invalid interpolation (\"\\U\") of 8bit characte", "%s:%d: неправильна інтерполяція (\"\\U\") 8-бітного символу \"%c"),
("inkscape", "All these settings depend on the plotter you use, fo", "Значення усіх цих параметрів залежать від використаного вами"),
("gas", "<fpu name> assemble for FPU architecture <fpu name>", "<назва fpu> зібрати для архітектури FPU <назва fpu>"),
("gnupg2", "no writable keyring found: %s", "не виявлено придатного до запису сховища ключів: %s"),
("bfd", "unknown attribute for symbol `%s': 0x%02x", "невідомий атрибути символу «%s»: 0x%02x"),
("flatpak", "Print version information and exit", "Показати дані щодо версії і завершити роботу"),
("NetworkManager", "The time, in seconds since the Unix Epoch, that the ", "Час, у секундах, з початку епохи UNIX, коли з'єднання було в"),
("binutils", "The options are: -i --input=<file> Name input file -", "Параметри: -i --input=<файл> Назва вхідного файла -o --outpu"),
("gnupg2", "the given preferred keyserver URL is invalid", "вказана адреса основного сервера ключів є некоректною"),
("glib20", "Include phony targets in the generated dependency fi", "Включити фіктивні призначення у створений файл залежностей"),
("elfutils", "cannot get ELF header '%s': %s", "не вдалося отримати заголовок ELF «%s»: %s"),
("e2fsprogs", "bad error behavior - %s", "помилкова поведінка у відповідь на помилку: %s"),
("libc", "Protocol family not supported", "Сімейство протоколів не підтримується"),
("zypper", "Repository '%s' not found by alias, number or URI.", "Сховище \"%s\" не знайдено за його псевдонімом, номером або UR"),
("elfutils", "Ignore permutation of buckets in SHT_HASH section", "Ігнорувати переставляння блоків у розділі SHT_HASH"),
("git", "'git help config' for more information", "\"git help config\" для додаткової інформації"),
("binutils", "Offset Info Type Sym. Value Symbol's Name", "Зсув Інфо Тип Знач.симв Назва символу"),
("sane-backends", "Enable double feed error due to paper thickness", "Увімкнути помилку подвійного подавання через товщину паперу"),
("ld", "Set address of named section", "встановити адресу іменованого розділу"),
("flatpak", "Warning: Omitting related ref ‘%s’ because it is not", "Попередження: пропускаємо пов'язане сховище «%s», оскільки й"),
("e2fsprogs", "Warning: the fs_type %s is not defined in mke2fs.con", "Попередження: fs_type для %s у mke2fs.conf не визначено"),
("NetworkManager", "Warning: argument '%s' is duplicated.", "Попередження: аргумент «%s» дубльовано."),
("openconnect", "STRAP signature failed", "Не вдалося виконати підписування STRAP"),
("gcc", "Source-expr at %L must be scalar or have the same ra", "Вихідний вираз на %L повинен бути скалярним або мати такий ж"),
("libvirt", "missing entry in supported dump formats", "у списку підтримуваних форматів дампів пропущено запис"),
("elfutils", "section [%2d] '%s': extra %llu bytes after last note", "розділ [%2d] «%s»: додаткові %llu байтів після останньої нот"),
("binutils", "cannot read symbol aux entry", "не вдалося прочитати допоміжний запис символу"),
("libgphoto2-6", "Could not load required camera driver '%s' (%s).", "Не вдалося завантажити драйвер фотоапарата '%s' (%s)."),
("gold", "linking with --incremental-full", "компонування за допомогою --incremental-full"),
("postgres-15", "the database system is shutting down", "система бази даних завершує роботу"),
("NetworkManager", "Error: Option '--terse' is specified the second time", "Помилка: параметр «--terse» вказано двічі."),
("coreutils", "failed to change group of %s from %s to %s", "не вдалося змінити групу %s з %s на %s"),
("cryptsetup", "Unknown crypt device type %s requested.", "Надіслано запит щодо невідомого типу пристрою шифрування, %s"),
("elfutils", "hash section [%2zu] '%s' does not contain enough dat", "розділ хешу [%2zu] «%s» містить недостатньо даних"),
("postgres-15", "table access method \"%s\" does not exist", "табличного методу доступу \"%s\" не існує"),
("e2fsprogs", "(There are %N @is containing @m @bs.)", "(Існує %N @i, що містять блоки кратного використання.)"),
("texinfo", "macro `%s' called with too many args", "макровизначення «%s» викликано з надмірною кількістю аргумен"),
("wxstd-3.2", "Failed to add custom font \"%s\".", "Не вдалося додати нетиповий шрифт «%s»."),
("elfutils", "while generating output file: %s", "під час спроби створення файла з виведеними даними: %s"),
("sane-backends", "Paper is pulled partly into ADF", "Папір частково втягнуто до протяжного пристрою"),
("wxstd-3.2", "Check to suppress hyphenation.", "Позначте, щоб придушити перенесення слів."),
("gnupg2", "enarmoring failed: %s", "помилка перетворення у формат ASCII: %s"),
("rpm", "build through %build (%prep, %conf, then compile) fr", "виконати аж до команди %build (%prep, %conf, потім збирання)"),
("binutils", "OpenBSD ELF auxiliary vector data", "Допоміжні векторні дані ELF OpenBSD"),
("virt-manager", "Configure a guest input device. Ex: --input tablet -", "Налаштувати пристрій введення гостьової системи. Приклад: --"),
("binutils", ", <unknown AMDGPU GPU type: %#x>", ", <невідомий тип графічного процесора AMDGPU: %#x>"),
("NetworkManager", "A connection with ovs-interface.type '%s' setting a ", "З'єднання з типом «%s» ovs-interface.type встановлює парамет"),
("gas", "-mmuladd Mark generated file as using multiply add/s", "-mmuladd позначити створений файл як такий, що використовує "),
("libvirt", "Could not retrieve 'bonding/arp_interval' for '%1$s'", "Не вдалося отримати «bonding/arp_interval» для «%1$s»"),
("rpm", "package scriptlets can be expanded at install time.", "допоміжні скрипти пакунка може бути розширено під час встано"),
("gnupg2", "Do you ultimately trust%%0A \"%s\"%%0Ato correctly cer", "Бажаєте встановити абсолютний рівень довіри до%%0A «%s»%%0Aз"),
("gnupg2", "invalid personal digest preferences", "некоректні особисті параметри контрольної суми"),
("gcc", "using complex absolute value function %qD when argum", "використання функції обчислення абсолютного значення комплек"),
("gas", "invalid unwind opcode", "некоректний код операції розгортання"),
("libvirt", "Failed to delete DiskDescriptor.xml of volume '%1$s'", "Не вдалося вилучити DiskDescriptor.xml тому «%1$s»"),
("postgres-15", "transform function must not return a set", "функція перетворення не повинна повертати набір"),
("sudoers", "internal error, %s overflow", "внутрішня помилка, переповнення %s"),
("gnupg2", "line %d: invalid date given", "рядок %d: вказано некоректну дату"),
("cryptsetup", "Bitmap options can be used only in bitmap mode.", "Параметри бітової карти можна використовувати лише у режимі "),
("libvirt", "Unable to remove bridge %1$s", "Не вдалося вилучити місток %1$s"),
("libc", "Accessing a corrupted shared library", "Доступ до пошкодженої бібліотеки спільного використання"),
("libvirt", "Failed to get leases info for %1$s", "Не вдалося отримати дані щодо надання для %1$s"),
("openconnect", "Could not send packet through Wintun adapter '%S': %", "Не вдалося надіслати пакет крізь адаптер Wintun «%S»: %s"),
("ld", "%P: %pB: warning: common of `%pT' overriding smaller", "%P: %pB: попередження: common «%pT» перевизначає менший comm"),
("postgres-15", "Expected \":\", but found \"%s\".", "Очікувалось \":\", але знайдено \"%s\"."),
("grub", "Load FreeBSD kernel module.", "Завантажити модуль ядра FreeBSD."),
("flatpak", "Updating appstream data for remote %s", "Оновлюємо дані appstream для віддаленого сховища %s"),
("ld", "%F%P: invalid --dsbt-index %s", "%F%P: некоректне значення --dsbt-index %s"),
("gcc", "Perform straight-line strength reduction.", "Виконати зменшення сили прямолінійного рядка."),
("wxstd-3.2", "Can't set value of '%s'", "Не вдалося встановити значення «%s»"),
("wxstd-3.2", "can't open user configuration file '%s'.", "Не вдалося відкрити файл налаштувань «%s»."),
("rpm", "Failed to read spec file from %s", "Не вдалося прочитати файл spec з %s"),
("elfutils", "cannot create temp file '%s'", "не вдалося створити файл тимчасових даних «%s»"),
("inkscape", "Control 32x33x34x35 - <b>Ctrl+Alt+Click</b>: reset, ", "Керування 32⨯33⨯34⨯35 — <b>Ctrl+Alt+клацання</b>, щоб скинут"),
("libvirt", "Domain snapshot %1$s reverted", "Знімок домену %1$s повернуто до попереднього стану"),
("dnf", "argument {}: not allowed with argument {}", "аргумент {}: не можна використовувати разом із аргументом {}"),
("bfd", "%F%P: failed to create the second PLT .eh_frame sect", "%F%P: не вдалося створити другий розділ .eh_frame PLT"),
("openconnect", "Current password too long.", "Поточний пароль є надто довгим."),
("gold", "%s: total archive members: %u", "%s: загалом елементів архіву: %u"),
("virt-manager", "By default libvirt will refuse to migrate a VM for c", "Типово, libvirt відмовляє у перенесенні ВМ для певних конфіг"),
("libc", "File size limit exceeded", "Перевищено обмеження на розмір файла"),
("psql-15", "PSQL_PAGER, PAGER name of external pager program", "PSQL_PAGER, PAGER ім'я програми зовнішнього пейджеру"),
("flatpak", "Allow partial commits in the created repo", "Дозволити часткові внески до створеного сховища"),
("binutils", "Out of range symbol index: %u", "Індекс символу поза межами припустимого діапазону: %u"),
("bfd", "sorry: modtab, toc and extrefsyms are not yet implem", "вибечте, modtab, toc та extrefsyms для команд dysymtab у пот"),
("openconnect", "ERROR: Cannot initialize sockets", "Помилка: не вдалося ініціалізувати сокети"),
("e2fsprogs", "Journal superblock magic number invalid!", "Контрольна сума суперблоку журналу є некоректною!"),
("gnome-software", "Can request data from system services", "Може надсилати запити щодо даних служб системи"),
("postgres-15", "attribute %d of relation with OID %u does not exist", "атрибут %d відношення з OID %u не існує"),
("NetworkManager", "Usage: nmcli device delete { ARGUMENTS | help } ARGU", "Користування: nmcli пристрій delete { ПАРАМЕТРИ | help } ПАР"),
("gnupg2", "run import filters and export key immediately", "запустити фільтри імпортування та експортувати ключ негайно"),
("openconnect", "Fetched CSD stub for %s platform (size is %d bytes).", "Отримано фіктивний CSD для платформи %s (розмір — %d байтів)"),
("gnupg2", "card does not support digest algorithm %s", "карткою не підтримується алгоритм контрольних сум %s"),
("gtk30-properties", "Maximum time allowed between two clicks for them to ", "Максимальний час між двома клацаннями, щоб вважати їх одним "),
("texinfo", "l2h: rename %s as %s failed: %s", "l2h: спроба перейменування %s на %s завершилася невдало: %s"),
("gcc", "%qD used in its own initializer", "%qD використовується у своєму власному ініціалізаторі"),
("bfd", "%pB: warning: %s unsupported in shared mode", "%pB: попередження: підтримки %s у спільному режимі не передб"),
("libgphoto2-6", "Continuous Low-speed Shooting", "Неперервне низькошвидкісне знімання"),
("shadow", "%s: the %s configuration in %s will be ignored", "%s: налаштування %s у %s буде проігноровано"),
("inkscape", "Status line hint\u0004<b>%s</b>: drag to make smooth, hov", "<b>%s</b>: перетягніть, щоб згладити; наведіть вказівник, що"),
("grub", "Load FreeDOS kernel.sys.", "Завантажити kernel.sys FreeDOS."),
("libvirt", "Too many servers '%1$d' for limit '%2$d'", "Забагато серверів, «%1$d», максимальна ж кількість — «%2$d»"),
("bfd", "%pB:%pA: error: relocation references symbol %s whic", "%pB:%pA: помилка: пересування посилається на символ %s, який"),
("libgphoto2-6", "Natural light auto White Balance Bias", "Природне освітлення, автоухил на баланс білого"),
("virt-manager", "Couldn't create default storage pool '%(path)s': %(e", "Не вдалося створити типове резервне сховище даних «%(path)s»"),
("postgres-15", "Sets the Bonjour service name.", "Встановлює ім'я служби Bonjour."),
("git", "(for porcelains) forget saved unresolved conflicts", "(для високорівневих команд) забути збережені невирішені конф"),
("ld", "%X%P: can not find overlays: %E", "%X%P: не вдалося знайти накладки: %E"),
("texinfo", "@%s arg must be `separate' or `end', not `%s'", "аргументом @%s має бути «separate» або «end», але не «%s»"),
("e2fsprogs", "The -T option may only be used once", "Параметр -T можна використовувати лише один раз"),
("libvirt", "Cannot set interface MAC on '%1$s'", "Не вдалося встановити MAC-адресу інтерфейсу на «%1$s»"),
("gnome-software", "Installed software is incompatible with %s, and will", "Встановлене програмне забезпечення є несумісним із %s. Його "),
("bfd", "warning: %pB: local symbol `%s' has no section", "попередження: %pB: локальний символ «%s» не має розділу"),
("glib20", "Element <%s> not allowed inside <%s>", "Елемент <%s> не може бути всередині <%s>"),
("dnf", "Temporarily disable active repositories for the purp", "Тимчасово вимкнути активні сховища для виконання поточної ко"),
("NetworkManager", "If specified, the password used with magic-packet-ba", "Якщо вказано, пароль, що використовується із Wake-on-LAN, за"),
("flatpak", "Search for remote apps/runtimes", "Шукати віддалені програми або сховища"),
("coreutils", "failed to set new range: %s", "помилка при встановленні нового діапазону: %s"),
("gas", "emulations not handled in this configuration", "у цій конфігурації не передбачено підтримки емуляції"),
("grub", "%s is deprecated. Use set gfxpayload=%s before linux", "%s вважається застарілим. Замість нього скористайтеся set gf"),
("sssd", "Whether to use Token-Groups", "Визначає, чи слід використовувати крупи реєстраційних записі"),
("psql-15", "set the session user identifier and the current user", "встановити ідентифікатор користувача сесії й ідентифікатор п"),
("gold", "unsupported file: 64-bit, little-endian", "непідтримуваний файл: 64-бітовий, прямий порядок байтів"),
("git", "strbuf_readlink('%s') failed", "strbuf_readlink(\"%s\") завершився невдало"),
("gas", "Invalid syntax in External addressing mode", "Некоректна синтаксична конструкція у режимі зовнішнього адре"),
("libc", "verification failed", "спроба перевірки зазнала невдачі"),
("gcc", "-fdiagnostics-column-origin=<number> Set the number ", "-fdiagnostics-column-origin=<число> Встановити номер першого"),
("gas", "start address not supported", "підтримки початкової адреси не передбачено"),
("e2fsprogs", "Illegal inode number passed to ext2fs_test_inode_bit", "ext2fs_test_inode_bitmap передано некоректну кількість inode"),
("gcc", "%<_Atomic%>-qualified parameter type %qT of %q+D", "параметр типу %qT з кваліфікатором %<_Atomic%> у %q+D"),
("ld", "Include all objects from following archives", "включити всі об’єкти з вказаних нижче архівів"),
("libgphoto2-6", "Get Vendor Extension Maps", "Отримати карти розширень виробника"),
("zypper", "modifyservice (ms) <OPTIONS> <ALIAS|#|URI>", "modifyservice (ms) <ПАРАМЕТРИ> <ПСЕВДОНІМ|#|URL>"),
("libc", "cannot change memory protections", "зміна захисту області пам’яті неможлива"),
("glib20", "Generate output in the format selected for by the ta", "Генерувати результат у форматі, який відповідає суфіксу назв"),
("shadow", "failed to create backup file", "не вдалося створити файл резервної копії"),
("glib20", "Unexpected lack of content trying to read a line", "Неочікувана відсутність вмісту при читанні рядка"),
("dnf", "shows results that requires, suggests, supplements, ", "показує результати, які надаються requires, suggests, supple"),
("gtk30-properties", "Type of bevel around toolbar buttons", "Тип фаски навколо кнопок пенала"),
("coreutils", "Floating point exception", "Помилка обчислень з рухомою комою"),
("libvirt", "no USB product ID supplied for '%1$s'", "не вказаний ID продукту USB для «%1$s»"),
("shadow", "%s: Can't get unique system GID (%s). Suppressing ad", "%s: не вдалося отримати унікальний загальносистемний GID (%s"),
("openconnect", "Server '%s' requested Basic authentication which is ", "Сервер «%s» вимагає базового розпізнавання, яке типово вимкн"),
("libc", "%s: unknown character in equivalent definition name", "%s: невідомий символ у назві еквівалентного визначення"),
("sssd", "Active Directory backup server address", "Адреса резервного сервера Active Directory"),
("gnupg2", "can't check signature with unsupported message-diges", "неможливо перевірити підпис із непідтримуваним алгоритмом ст"),
("glib20", "Error parsing parameter %d: %s", "Сталася помилка під час обробки параметра %d: %s"),
("wget", "--ignore-length ignore 'Content-Length' header field", "--ignore-length ігнорувати поле заголовку `Content-Length'"),
("ld", "%F%P: final link failed: %E", "%F%P: спроба остаточного компонування зазнала невдачі: %E"),
("binutils", "unrecognized symbol flag `%s'", "нерозпізнаний прапорець символу «%s»"),
("gettext-tools", "-t, --to-code=NAME encoding for output", "-t, --to-code=НАЗВА кодування виводу"),
("NetworkManager", "missing prefix length for %s '%s', defaulting to %d", "пропущено префікс довжини для %s «%s», повертаємося до типов"),
("zypper", "The following query does not lock anything:", "Наступний запит нічого не блокує:"),
("inkscape", "Ignore this word only once", "Ігнорувати це слово лише у цьому випадку"),
("sssd", "PAM service names that map to the GPO (Deny)ServiceL", "Назви служб PAM, які виконують прив’язування до параметрів п"),
("gnome-software", "Operating System Updates Unavailable", "Оновлення операційної системи недоступні"),
("git", "git commit-graph write [--object-dir <dir>] [--appen", "git commit-graph write [--object-dir <директорія>] [--append"),
("libc", "Start NUMBER threads", "Запустити вказане ЧИСЛО потоків обробки"),
("texinfo", "in @%s empty cross reference name after expansion `%", "у @%s виявлено порожню назву перехресного посилання після ро"),
("gcc", "invalid definition of qualified type %qT", "неприпустиме визначення кваліфікованого типу %qT"),
("gcc", "%<symver%> attribute only applies to functions and v", "атрибут %<symver%> застосовується тільки до функцій та змінн"),
("git", "Show changes between commits, commit and working tre", "Показати зміни між комітами, комітом та робочим деревом, тощ"),
("anaconda", "Erasing the data cannot be undone. Be sure to have b", "Наслідки дії з витирання даних є незворотними. Не забудьте с"),
("elfutils", "section [%2zu] '%s' must not be writable", "розділ [%2zu] «%s» не повинен бути придатним до запису"),
("git", "could not parse HEAD commit", "не вдалося розібрати HEAD коміт"),
("sudoers", "Apply defaults in the target user's login class if t", "Застосовувати типові параметри у класі вказаного користувача"),
("selinux-python", "Could not get module priority", "Не вдалося отримати рівень пріоритетності модуля"),
("gtk30-properties", "Display the standard forward arrow button", "Показувати стандартну кнопку із стрілкою вперед"),
("glib20", "Cannot truncate GBufferedInputStream", "Не вдалося урізати GMemoryInputStream"),
("gettext-tools", "The -l and -d options are mandatory. The .dll file i", "Параметри -l та -d є обов'язковими. Файл .dll шукається у пі"),
("git", "Display help information about Git", "Відобразити довідкову інформацію про Git"),
("gettext-tools", "--escape use C escapes in output, no extended chars", "--escape використовувати у виводі екранування у стилі C, без"),
("gtk30-properties", "Keybinding to activate the menu bar", "Клавіша для активації панелі меню"),
("glib20", "Destination name to monitor", "Назва призначення для спостерігання"),
("git", "abort if fast-forward is not possible", "перервати, якщо перемотування вперед неможливе"),
("psql-15", "Pager is used for long output.", "Пейджер використовується для виведення довгого тексту."),
("git", "write_reuse_object: could not locate %s, expected at", "write_reuse_object: не вдалося знайти %s, очікуваний зі зміщ"),
("gas", "file finished with an open IT block.", "файл завершено незавершеним блоком IT."),
("gas", ".error directive invoked in source file", "у файлі початкового коду викликано директивну .error"),
("sudoers", "Log user's input for the command being run", "Записувати дані, вказані користувачем під час виконання кома"),
("sudoers", "Enable sudoers netgroup support", "Увімкнути підтримку мережевих груп у sudoers"),
("wget", "File %s retrieved but checksum does not match.", "Файл %s отримано, але його контрольна сума не збігається із "),
("gtk30-properties", "Positioning hints for when the menu might fall off-s", "Розташування підказок у випадку, коли меню може вийти за меж"),
("openconnect", "Set path of initial request URL", "Встановити шлях для адреси початкового запиту"),
("zypper", "Repository '%s' not found by its alias, number, or U", "Сховище \"%s\" не знайдено за його псевдонімом або URI."),
("postgres-15", "Enables the planner's use of merge join plans.", "Дає змогу планувальнику використовувати плани з'єднання об'є"),
("coreutils", "--block-signal[=SIG] block delivery of SIG signal(s)", "--block-signal[=СИГНАЛ] блокувати доставлення сигналів СИГНА"),
("libgphoto2-6", "Fluorescent: Tungsten", "Флуоресцентна лампа: лампа розжарювання"),
("e2fsprogs", "Wrong magic number --- RESERVED_15", "Помилкова контрольна сума --- RESERVED_15"),
("gas", "expecting lockable instruction after `lock'", "очікуємо на придатну до блокування інструкцію після «lock»"),
("wget", "Wrote HTML-ized index to %s.", "Покажчик у форматі HTML записано до файла %s."),
("dnf", "%s: using metadata from %s.", "%s: з використанням метаданих з %s."),
("virt-manager", "<span size='small'>Cloning does <u>not</u> alter the", "<span size='small'>Клонування <u>не</u> змінює вмісту образу"),
("zypper", "Repository '%s' has been successfully enabled.", "Сховище \"%s\" було успішно увімкнено."),
("cryptsetup", "Cannot initialize device-mapper, running as non-root", "Не можна ініціалізувати device-mapper, якщо програму запущен"),
("bfd", "%s has both normal and TLS relocs", "%s містить одразу звичайні пересування і пересування TLS"),
("git", "corrupt ewah bitmap: commit index %u out of range", "пошкоджений ewah bitmap: індекс коміту %u поза межами діапаз"),
("gold", "read-only segment has dynamic relocations", "сегмент, призначений лише для читання, містить динамічні пер"),
("postgres-15", "invalid encoding name \"%s\"", "неприпустиме ім’я кодування \"%s\""),
("libgphoto2-6", "Camera identification: Model: %s Owner: %s Power sta", "Визначення фотоапарата: Модель: %s Власник: %s Живлення: %s "),
("anaconda", "To access the Red Hat CDN, a valid Red Hat subscript", "Для доступу до Red Hat CDN потрібна чинна передплата Red Hat"),
("shadow", "%s: failed to copy the lastlog entry of user %lu to ", "%s: не вдалося скопіювати запис lastlog користувача %lu до к"),
("libgphoto2-6", "Single: Silent shooting", "Однокадрове знімання: тихе знімання"),
("gtk30-properties", "TRUE if the origin of the context should be at the c", "TRUE, якщо джерело контексту має бути у куті сторінки, а не "),
("gcc", "Junk after MAP statement at %C", "Сміття після оператора MAP на %C"),
("selinux-python", "Could not set user in file context for %s", "Не вдалося вказати користувача у контексті файла для %s"),
("grub", "You need to specify at least one command.", "Вам слід вказати принаймні одну команду."),
("gtk30-properties", "The minimum number of children to allocate consecuti", "Мінімальна кількість дочірніх елементів, які можна розміщува"),
("inkscape", "Select all objects or all nodes", "Позначити всі об'єкти чи всі вузли"),
("coreutils", "eof CHAR CHAR will send an end of file (terminate th", "eof СИМВОЛ СИМВОЛ буде означати кінець файла (припинення вве"),
("inkscape", "Render to JPEG file format", "Обробити до формату файлів JPEG"),
("sssd", "Enables principal canonicalization", "Вмикає перетворення реєстраційних записів у канонічну форму"),
("sane-backends", "Sets lamp on/off", "Визначає стан вмикання/вимикання лампи"),
("postgres-15", "too many lexemes in thesaurus entry", "занадто багато лексем в елементі тезаурусу"),
("inkscape", "Print debugging information and exit", "Вивести діагностичні дані і завершити роботу"),
("postgres-15", "skipping vacuum of \"%s\" --- lock not available", "очистка \"%s\" пропускається --- блокування недоступне"),
("coreutils", "Written by %s, %s, %s, %s, %s, and %s.", "Автор програми %s, %s, %s. %s, %s та %s"),
("coreutils", "-c --format=FORMAT use the specified FORMAT instead ", "-c --format=ФОРМАТ використовувати вказаний ФОРМАТ, а не тип"),
("zypper", "Repository '%s' is out of date. Running '%s' might h", "Дані сховища \"%s\" застаріли. Запуск \"%s\" має допомогти."),
("appstream", "This desktop-entry file has the 'Hidden' property se", "Для цього файла запису desktop встановлено властивість «прих"),
("selinux-python", "You must add at least one role for %s", "Треба додати принаймні одну роль для %s"),
("coreutils", "Write each FILE to standard output, with line number", "Вивести кожне ФАЙЛ до стандартного виведення із додаванням н"),
("grub", "Embedding is not possible. GRUB can only be installe", "Вбудовування неможливе. GRUB можна встановити на таку конфіг"),
("flatpak", "While trying to resolve ref %s:", "Під час спроби визначити посилання %s:"),
("appstream", "Users are encouraged to purchase specific real-world", "Користувачам пропонують придбати певні матеріальні товари"),
("NetworkManager", "Team JSON configuration [none]", "Налаштування JSON команди [немає]"),
("gtk30-properties", "Which IM module should be used", "Модуль вводу, що використовується"),
("binutils", "Displaying the debug contents of section %s is not y", "Відображення діагностичної інформації розділу %s ще не підтр"),
("cryptsetup", "This BITLK device is in an unsupported state and can", "Цей пристрій BITLK перебуває у непідтримуваному стані — його"),
("psql-15", "\\set [NAME [VALUE]] set internal variable, or list a", "\\set [NAME [VALUE]] встановити внутрішню змінну або вивести "),
("gcc", "cannot convert %qE to %qT because", "не вдається перетворити %qE в %qT, оскільки"),
("wxstd-3.2", "Shows a preview of the paragraph settings.", "Показує попередній перегляд параметрів абзацу."),
("coreutils", "-W, --word-regexp=REGEXP use REGEXP to match each ke", "-W, --word-regexp=REGEXP регулярний вираз для ключових слів "),
("ld", "-z nocopyreloc Don't create copy relocs", "-z nocopyreloc не створювати переміщено копій"),
("ld", "--enable-stdcall-fixup Link _sym to _sym@nn without ", "--enable-stdcall-fixup компонувати _sym з _sym@nn без попере"),
("inkscape", "Remove Inkscape-specific SVG attributes/properties", "Вилучити специфічні для Inkscape атрибути і властивості SVG"),
("inkscape", "<span weight=\"bold\" size=\"larger\">Save changes to do", "<span weight=\"bold\" size=\"larger\">Зберегти перед закриванням"),
("anaconda", "Checking software dependencies...", "Перевіряємо залежності програмного забезпечення…"),
("wxstd-3.2", "Monitoring individual files for changes is not suppo", "Спостереження за змінами у окремих файлах у поточній версії "),
("openconnect", "Using SWTPM due to TPM_INTERFACE_TYPE environment va", "Використовуємо SWTPM через встановлення змінної середовища T"),
("gettext-tools", "--strict write strict uniforum style", "--strict виводити у точній відповідності до uniforum"),
("NetworkManager", "empty text does not describe a rule", "порожній текст не описує правило"),
("anaconda", "Protocol in URL does not match selected protocol", "Протокол у адресі не відповідає вибраному протоколу"),
("gas", "-mCPU equivalent to -march=CPU -mtune=CPU. Deprecate", "-mCPU еквівалент -march=CPU -mtune=CPU. Застарілий. -no-mCPU"),
("texinfo", "malformed variable assignment: %s", "помилкове форматування у виразі визначення змінної: %s"),
("virt-manager", "No hypervisor options were found for this connection", "Для цього з'єднання не було знайдено жодного параметра гіпер"),
("cryptsetup", "LUKS: Default keysize with XTS mode (two internal ke", "LUKS: типовий розмір ключа у режимі XTS (два вбудованих ключ"),
("anaconda", "You must include a PReP Boot Partition within the fi", "Вам слід включити розділ завантаження PReP у перших 4 ГіБ фо"),
("grub", "other software is using the embedding area, and ther", "область вбудовування використано іншим програмним забезпечен"),
("e2fsprogs", "Permission denied to resize filesystem", "Недостатньо прав доступу для зміни розмірів файлової системи"),
("gold", "Load a plugin library", "завантажити бібліотеку додатків"),
]

print("рядків у запасному корпусі:", len(FALLBACK))


In [ ]:
docs = load_system_corpus()
FULL_CORPUS = len(docs) > 50000        # чи це справжній корпус, а не запасний

if not FULL_CORPUS:
    print("⚠️  українська локаль не знайдена (або надто мала) — беремо запасний корпус.")
    print("    Числа нижче будуть інші, ніж у лекції, але весь код виконається.")
    docs = FALLBACK

programs = collections.Counter(program for program, src, dst in docs)
print("документів:", len(docs))
print("програм:   ", len(programs))
print("приклад:   ", docs[0][0], "|", docs[0][1][:44], "|", docs[0][2][:44])


## 3 · Задача: мітка з оригіналу, ознаки з перекладу

Мітка «це повідомлення про помилку чи ні» береться з **англійського** оригіналу
простою регуляркою. Ознаки модель бачить тільки з **українського** перекладу.

Так замір не стає круговим: модель не бачить того, з чого зроблено мітку. Той
самий прийом уживала [тема 03](../03-morphology/lecture.html), і на ньому ж
стоїть [тема 06](../06-text-classification/lecture.html).


In [ ]:
ERROR_PATTERN = re.compile(
    r'\b(error|failed|cannot|could not|unable|invalid|denied|no such)\b', re.I)

labels = np.array([1 if ERROR_PATTERN.search(src) else 0 for program, src, dst in docs])
texts = [dst for program, src, dst in docs]

print("документів:          ", len(texts))
print("«помилка» (клас 1):  ", int(labels.sum()))
print("«не помилка» (клас 0):", int((labels == 0).sum()))
print("частка класу 1:       %.4f" % labels.mean())


## 4 · Поділ, ознаки, модель

Правило поділу одне на всю тему: **70 % на навчання, 30 % на перевірку,
поділ стратифікований** (частка класів у обох частинах однакова).

`TfidfVectorizer` навчається **тільки на навчальній частині**. Це не дрібниця:
якщо показати йому перевірні документи, словник і ваги IDF почнуть знати про них
наперед, і замір стане завищеним.

Токенізатор — канонічний для всього курсу, той самий, що в темах 02-06.


In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"

SPLIT_CACHE = {}

def split_and_vectorize(seed):
    """Один поділ корпусу на 70/30 і одна матриця TF-IDF, навчена на train.
    Результат кешується: той самий поділ знадобиться далі ще раз."""
    if seed in SPLIT_CACHE:
        return SPLIT_CACHE[seed]
    text_train, text_test, y_train, y_test = train_test_split(
        texts, labels, test_size=0.3, random_state=seed, stratify=labels)
    vectorizer = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
    X_train = vectorizer.fit_transform(text_train)   # словник рахується тут…
    X_test = vectorizer.transform(text_test)         # …а тут лише застосовується
    SPLIT_CACHE[seed] = (X_train, X_test, y_train, y_test)
    return SPLIT_CACHE[seed]

X_train, X_test, y_train, y_test = split_and_vectorize(0)

# стінний час на спільній машині показує чергу, а не роботу, тож міряємо процесорний
started = time.process_time()
model = LogisticRegression(max_iter=1000, random_state=0).fit(X_train, y_train)
fit_seconds = time.process_time() - started
predicted = model.predict(X_test)

print("ознак у словнику:  ", X_train.shape[1])
print("навчальних докум.: ", X_train.shape[0])
print("перевірних докум.: ", X_test.shape[0])
print("з них клас 1:      ", int(y_test.sum()))
print("навчання, процесорних секунд: %.2f" % fit_seconds)


## 5 · Матриця плутанини — чотири числа, з яких усе

Модель відповіла на кожен перевірний документ. Порівняння відповіді з істиною дає
рівно чотири випадки, і більше нічого:

| | модель каже «0» | модель каже «1» |
|---|---|---|
| **насправді 0** | TN — правильно відкинула | FP — хибна тривога |
| **насправді 1** | FN — пропустила | TP — правильно знайшла |

Усе інше в цій темі — комбінації цих чотирьох чисел.


In [ ]:
matrix = confusion_matrix(y_test, predicted, labels=[0, 1])
TN, FP, FN, TP = matrix.ravel()

print("            модель «0»   модель «1»")
print("насправді 0  TN %6d   FP %6d" % (TN, FP))
print("насправді 1  FN %6d   TP %6d" % (FN, TP))
print()
print("сума чотирьох чисел:", TN + FP + FN + TP, "= перевірних документів")


### Усі метрики руками з цих чотирьох чисел

- **`precision`** класу 1: серед тих, кого модель назвала одиницею, скільки
  справді одиниці. `TP / (TP + FP)`.
- **`recall`** класу 1: серед справжніх одиниць скільки модель знайшла.
  `TP / (TP + FN)`.
- **F1** — гармонійне середнє `precision` і `recall`: `2·P·R / (P + R)`.
- **Частка правильних (accuracy)** — `(TP + TN) / усе`.

Клас 0 рахується так само, тільки ролі міняються місцями: для нього «своїми» є
TN, а помилками — FN (хибна тривога навпаки) і FP (пропуск навпаки).


In [ ]:
def metrics_by_hand(tn, fp, fn, tp):
    """precision, recall і F1 для класу 1 — просто з чотирьох чисел."""
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

P1, R1, F1_pos = metrics_by_hand(TN, FP, FN, TP)
# для класу 0 «своїми» стають TN, а помилками — FN і FP
P0, R0, F1_neg = metrics_by_hand(TP, FN, FP, TN)
accuracy_by_hand = (TP + TN) / (TN + FP + FN + TP)

print("клас 1 (помилка):     P %.4f  R %.4f  F1 %.4f  n %d" % (P1, R1, F1_pos, TP + FN))
print("клас 0 (не помилка):  P %.4f  R %.4f  F1 %.4f  n %d" % (P0, R0, F1_neg, TN + FP))
print("частка правильних:    %.4f" % accuracy_by_hand)

# перевірка: те саме має дати бібліотека
lib_P, lib_R, lib_F, lib_n = precision_recall_fscore_support(
    y_test, predicted, labels=[0, 1], zero_division=0)
assert np.allclose([P0, P1], lib_P), "precision розійшовся зі sklearn"
assert np.allclose([R0, R1], lib_R), "recall розійшовся зі sklearn"
assert np.allclose([F1_neg, F1_pos], lib_F), "F1 розійшлася зі sklearn"
assert math.isclose(accuracy_by_hand, accuracy_score(y_test, predicted))
print("✅ збігається зі sklearn до останнього знака")


In [ ]:
# те саме одним рядком — так його друкують у звітах
print(classification_report(y_test, predicted, digits=4,
                            target_names=["не помилка", "помилка"]))


## 6 · Модель, яка не вміє нічого

Найдешевша перевірка будь-якого числа: скільки набере модель, яка **завжди**
відповідає найбільшим класом. Вона нічого не рахує й нічого не знає.


In [ ]:
always_zero = np.zeros_like(y_test)      # «усе — не помилка»

dumb_accuracy = accuracy_score(y_test, always_zero)
dumb_macro = f1_score(y_test, always_zero, average='macro', zero_division=0)
dumb_f1_pos = f1_score(y_test, always_zero, pos_label=1, zero_division=0)

print("дурна модель «усе — не помилка»")
print("  частка правильних: %.4f" % dumb_accuracy)
print("  макро-F1:          %.4f" % dumb_macro)
print("  F1 класу «помилка» %.4f" % dumb_f1_pos)
print()
print("справжня модель")
print("  частка правильних: %.4f" % accuracy_score(y_test, predicted))
print("  макро-F1:          %.4f" % f1_score(y_test, predicted, average='macro'))
print()
print("розрив у частці правильних: %.4f" % (accuracy_score(y_test, predicted) - dumb_accuracy))
print("розрив у макро-F1:          %.4f" % (f1_score(y_test, predicted, average='macro') - dumb_macro))


Частка правильних у дурної моделі береться з нічого: вона дорівнює частці
найбільшого класу. Формула для макро-F1 дурної моделі теж закрита — перевіримо
її на числах.


In [ ]:
share_of_class_1 = y_test.mean()

# «усе — клас 0»: для класу 0 recall дорівнює 1, precision — частці класу 0
predicted_share = 1 - share_of_class_1
f1_of_class_0 = 2 * predicted_share * 1.0 / (predicted_share + 1.0)
formula_macro = f1_of_class_0 / 2          # F1 класу 1 дорівнює нулю

print("частка класу 1:                %.4f" % share_of_class_1)
print("частка правильних за формулою: %.4f" % predicted_share)
print("макро-F1 за формулою:          %.4f" % formula_macro)
assert math.isclose(formula_macro, dumb_macro, abs_tol=1e-12), "формула не збіглася"
assert math.isclose(predicted_share, dumb_accuracy, abs_tol=1e-12)
print("✅ формула збіглася із заміром")


## 7 · Три способи усереднити метрику по класах

Класів два, метрик по класах теж два комплекти. Щоб дістати **одне** число, їх
усереднюють, і способів три:

- **макро** — просте середнє F1 по класах; кожен клас має один голос незалежно
  від розміру;
- **зважене (weighted)** — середнє F1, зважене на кількість документів у класі;
- **мікро** — спершу складають TP, FP і FN **по всіх класах разом**, і тільки
  потім рахують одну F1.


In [ ]:
macro = f1_score(y_test, predicted, average='macro')
weighted = f1_score(y_test, predicted, average='weighted')
micro = f1_score(y_test, predicted, average='micro')
accuracy = accuracy_score(y_test, predicted)

print("макро-F1:          %.4f" % macro)
print("зважена F1:        %.4f" % weighted)
print("мікро-F1:          %.4f" % micro)
print("частка правильних: %.4f" % accuracy)
print()
print("макро як середнє двох F1: %.4f" % ((F1_neg + F1_pos) / 2))
print("різниця мікро й частки правильних: %.20f" % (micro - accuracy))


## 8 · Мікро-F1 **дорівнює** частці правильних. Доведення

Візьмімо однокласове передбачення: кожен приклад дістає рівно одну відповідь і має
рівно одну істинну мітку. Порахуймо, що кожен приклад додає до сум по всіх класах.

- Відповідь **правильна**: до `TP` свого класу додається одиниця, до `FP` і `FN`
  жодного класу — нічого.
- Відповідь **хибна**, модель сказала `p`, а насправді `t`: до `FP` класу `p`
  додається одиниця, до `FN` класу `t` — одиниця, до `TP` — нічого.

Отже, склавши по всіх класах: `ΣTP` = кількість правильних `C`,
`ΣFP` = `ΣFN` = кількість хибних `W`. Далі:

    мікро-precision = C / (C + W) = C / N
    мікро-recall    = C / (C + W) = C / N

Два числа **рівні**, а гармонійне середнє двох рівних чисел дорівнює їм самим.
Тому `мікро-F1 = C / N`, а це й є частка правильних. Перевіримо всі чотири рівності.


In [ ]:
def micro_counts(y_true, y_pred, class_list):
    """ΣTP, ΣFP, ΣFN по всіх класах — те, з чого рахується мікро-усереднення."""
    sum_tp = sum_fp = sum_fn = 0
    for one_class in class_list:
        sum_tp += int(((y_pred == one_class) & (y_true == one_class)).sum())
        sum_fp += int(((y_pred == one_class) & (y_true != one_class)).sum())
        sum_fn += int(((y_pred != one_class) & (y_true == one_class)).sum())
    return sum_tp, sum_fp, sum_fn

sum_tp, sum_fp, sum_fn = micro_counts(y_test, predicted, [0, 1])
correct = int((y_test == predicted).sum())
wrong = len(y_test) - correct

print("ΣTP %6d   правильних відповідей %6d" % (sum_tp, correct))
print("ΣFP %6d   хибних відповідей     %6d" % (sum_fp, wrong))
print("ΣFN %6d   хибних відповідей     %6d" % (sum_fn, wrong))
assert sum_tp == correct and sum_fp == wrong and sum_fn == wrong, "суми не збіглися"

micro_precision = sum_tp / (sum_tp + sum_fp)
micro_recall = sum_tp / (sum_tp + sum_fn)
micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall)
print()
print("мікро-precision    %.4f" % micro_precision)
print("мікро-recall       %.4f" % micro_recall)
print("мікро-F1           %.4f" % micro_f1)
print("частка правильних  %.4f" % accuracy)
assert micro_precision == micro_recall == micro_f1 == accuracy, "тотожність не виконалась"
assert f1_score(y_test, predicted, average='micro') == accuracy_score(y_test, predicted)
print("✅ мікро-precision = мікро-recall = мікро-F1 = частка правильних, рівно")


### Де ця рівність ламається

Доведення трималося на одному припущенні: **один приклад — рівно одна відповідь**.
У багатоміткових задачах (документ може мати кілька тем одразу) воно не виконується,
і мікро-F1 стає окремою метрикою, вартою своєї назви.

Маленький приклад на чотирьох прикладах і трьох мітках — рахується руками.


In [ ]:
# рядок — приклад, стовпчик — мітка; одиниця означає «мітка стоїть»
multi_true = np.array([[1, 1, 0],
                       [0, 1, 0],
                       [1, 0, 1],
                       [0, 0, 1]])
multi_pred = np.array([[1, 0, 0],
                       [0, 1, 1],
                       [1, 0, 1],
                       [0, 0, 1]])

multi_micro = f1_score(multi_true, multi_pred, average='micro')
# частка правильних для багатомітки — це частка рядків, збіглих ЦІЛКОМ
subset_accuracy = float(np.all(multi_true == multi_pred, axis=1).mean())

print("мікро-F1:                    %.4f" % multi_micro)
print("частка цілком правильних:    %.4f" % subset_accuracy)
assert multi_micro != subset_accuracy, "у багатомітковій задачі вони мали розійтись"
print("✅ тут вони РІЗНІ — тотожність тримається лише на однокласовому передбаченні")


## 9 · Природний дисбаланс: 266 програм

Двокласова задача була ще милосердна: 0.21 проти 0.79. Справжній дисбаланс лежить
поруч — у тому, з якої програми документ.


In [ ]:
sizes = sorted(programs.values(), reverse=True)
biggest_program, biggest_size = programs.most_common(1)[0]

print("програм:                    ", len(sizes))
print("найбільша (%s): %d документів" % (biggest_program, biggest_size))
print("найменша:                    %d документ(ів)" % sizes[-1])
print("відношення:                  %d×" % (sizes[0] // sizes[-1]))
print("медіана:                     %d" % int(np.median(sizes)))
print("топ-10 програм:              %.4f корпусу" % (sum(sizes[:10]) / len(docs)))
print("програм менш ніж по 50 док.: %d" % sum(1 for s in sizes if s < 50))
print()
print("десять найбільших:", programs.most_common(10))


### Багатокласова задача: вгадай програму за текстом

Беремо `K` найбільших програм і вчимо модель казати, з якої програми рядок.
Що більше `K`, то довший хвіст крихітних класів — і то далі розходяться макро й мікро.

Модель — `ComplementNB`: вона вчиться за частку секунди, а тема зараз не про модель,
а про метрику. Три зерна на кожну точку, як завжди в курсі.


In [ ]:
program_order = [p for p, _ in programs.most_common()]

def multiclass_run(K, seed):
    """Одна точка: K найбільших програм, стратифікований поділ, ComplementNB."""
    keep = set(program_order[:K])
    rows = [(program, dst) for program, src, dst in docs if program in keep]
    # клас із одним документом стратифікований поділ не поділить — прибираємо
    counts = collections.Counter(program for program, dst in rows)
    rows = [(program, dst) for program, dst in rows if counts[program] >= 2]

    y = np.array([program for program, dst in rows])
    X = [dst for program, dst in rows]
    text_train, text_test, y_tr, y_te = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y)
    vectorizer = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
    A = vectorizer.fit_transform(text_train)
    B = vectorizer.transform(text_test)
    p = ComplementNB().fit(A, y_tr).predict(B)

    micro_value = f1_score(y_te, p, average='micro')
    # та сама тотожність, але вже на сотнях класів
    assert micro_value == accuracy_score(y_te, p), "мікро розійшлось із часткою правильних"
    return len(set(y)), len(rows), micro_value, f1_score(y_te, p, average='macro')

K_LIST = (2, 3, 5, 8, 12, 20, 50, 120, 180, 266) if FULL_CORPUS else (2, 3, 5, 8, 12, 20, 40)

print("класів  докум.  мікро-F1        макро-F1        розрив")
multiclass_table = {}
for K in K_LIST:
    micro_seeds, macro_seeds = [], []
    for seed in (0, 1, 2):
        n_classes, n_rows, mi, ma = multiclass_run(K, seed)
        micro_seeds.append(mi)
        macro_seeds.append(ma)
    gap = np.mean(micro_seeds) - np.mean(macro_seeds)
    multiclass_table[n_classes] = (np.mean(micro_seeds), np.mean(macro_seeds))
    print("%6d  %6d  %.4f ±%.4f  %.4f ±%.4f  %+.4f"
          % (n_classes, n_rows, np.mean(micro_seeds), np.std(micro_seeds),
             np.mean(macro_seeds), np.std(macro_seeds), gap))


### Куди дівається вага макро-усереднення

Розрив між мікро й макро — не таємниця. Макро дає кожному класу **один голос**,
скільки б у ньому не було документів. Порахуймо, скільки голосів і скільки
документів має кожна група класів.


In [ ]:
rows_all = [(program, dst) for program, src, dst in docs if programs[program] >= 2]
y_all = np.array([program for program, dst in rows_all])
X_all = [dst for program, dst in rows_all]
t_tr, t_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.3,
                                          random_state=0, stratify=y_all)
vec_all = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
A_all = vec_all.fit_transform(t_tr)
B_all = vec_all.transform(t_te)
pred_all = ComplementNB().fit(A_all, y_tr).predict(B_all)

class_list = sorted(set(y_all))
P_all, R_all, F_all, N_all = precision_recall_fscore_support(
    y_te, pred_all, labels=class_list, zero_division=0)

print("класів %d, перевірних документів %d" % (len(class_list), len(y_te)))
print("мікро-F1 %.4f   частка правильних %.4f   макро-F1 %.4f   зважена F1 %.4f"
      % (f1_score(y_te, pred_all, average='micro'), accuracy_score(y_te, pred_all),
         f1_score(y_te, pred_all, average='macro'), f1_score(y_te, pred_all, average='weighted')))
print()

BIG, SMALL = (1000, 50) if FULL_CORPUS else (8, 5)
groups = {
    "великі":  [i for i, c in enumerate(class_list) if programs[c] >= BIG],
    "середні": [i for i, c in enumerate(class_list) if SMALL <= programs[c] < BIG],
    "малі":    [i for i, c in enumerate(class_list) if programs[c] < SMALL],
}
print("група     класів  голосів  документів  середня F1  F1 = 0 рівно")
for name, idx in groups.items():
    if not idx:
        continue
    print("%-9s %6d  %6.4f  %9.4f  %10.4f  %d"
          % (name, len(idx), len(idx) / len(class_list),
             sum(N_all[i] for i in idx) / len(y_te),
             float(np.mean([F_all[i] for i in idx])),
             sum(1 for i in idx if F_all[i] == 0)))


In [ ]:
# профіль кожного класу: розмір у корпусі, його F1 і скільки його документів
# у перевірній частині. Саме ці числа малює інтерактив 5 лекції.
class_profile = sorted([(int(programs[c]), round(float(F_all[i]), 4), int(N_all[i]))
                        for i, c in enumerate(class_list)], key=lambda pair: -pair[0])
print("розмір класу, F1, документів у перевірній частині — усі", len(class_profile), "класів:")
print(class_profile)


In [ ]:
# те саме зведено: скільки голосу й документів має хвіст при різних порогах «малого»
CUTS = (3, 5, 8, 10, 15, 20, 30, 50, 75, 100, 150, 200, 300, 500, 1000, 2000)
all_docs = sum(row[2] for row in class_profile)
print("поріг  малих  голосу  документів  F1 малих  F1 решти")
for cut in CUTS:
    small = [row for row in class_profile if row[0] < cut]
    rest = [row for row in class_profile if row[0] >= cut]
    if not small or not rest:
        continue
    print("%5d  %5d  %.4f  %10.4f  %8.4f  %8.4f"
          % (cut, len(small), len(small) / len(class_profile),
             sum(row[2] for row in small) / all_docs,
             float(np.mean([row[1] for row in small])),
             float(np.mean([row[1] for row in rest]))))


In [ ]:
# дурна модель у багатокласовій задачі: завжди називати найбільшу програму
always_biggest = np.full_like(y_te, biggest_program)
print("дурна модель «усе — %s»" % biggest_program)
print("  мікро-F1 (= частка правильних): %.4f" % f1_score(y_te, always_biggest, average='micro'))
print("  макро-F1:                       %.4f"
      % f1_score(y_te, always_biggest, average='macro', zero_division=0))


## 10 · Поріг — це ручка, а не властивість моделі

Логістична регресія повертає не мітку, а число від 0 до 1. Мітка виникає тільки
після порівняння з порогом, і `0.5` — просто значення за замовчуванням, а не
оптимум. Зсув порога рухає `precision` і `recall` **в протилежні боки**.


In [ ]:
scores = model.predict_proba(X_test)[:, 1]     # оцінка «це помилка» для кожного документа

def at_threshold(t):
    """Чотири числа матриці плутанини й метрики при заданому порозі."""
    guess = (scores >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, guess, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return tn, fp, fn, tp, precision, recall, f1, (tp + tn) / len(y_test)

print("поріг     FP     FN  precision  recall     F1     частка прав.")
for t in (0.10, 0.20, 0.27, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90):
    tn, fp, fn, tp, pr, rc, f1, acc = at_threshold(t)
    print("%.2f  %6d %6d    %.4f   %.4f  %.4f    %.4f" % (t, fp, fn, pr, rc, f1, acc))


In [ ]:
# повна крива: 101 поріг з кроком 0.01. Саме ці числа малює лекція.
grid = [round(i / 100, 2) for i in range(101)]
curve = [at_threshold(t) for t in grid]
tp_curve = [int(c[3]) for c in curve]     # int(), щоб надрукувалось як число, а не як np.int64
fp_curve = [int(c[1]) for c in curve]

best_f1_at = max(range(101), key=lambda i: curve[i][6])
best_acc_at = max(range(101), key=lambda i: curve[i][7])

print("найкраща F1 класу «помилка»: поріг %.2f, F1 %.4f (при 0.50 було %.4f)"
      % (grid[best_f1_at], curve[best_f1_at][6], curve[50][6]))
print("найбільша частка правильних: поріг %.2f, %.4f (при 0.50 було %.4f)"
      % (grid[best_acc_at], curve[best_acc_at][7], curve[50][7]))
print()
print("TP по порогах:", tp_curve)
print("FP по порогах:", fp_curve)


### «Найкращий поріг» задає ціна помилки, а не дані

Пропуск (FN) і хибна тривога (FP) коштують по-різному, і цю різницю задає задача,
а не модель. Порахуймо втрату `FP + ціна × FN` і знайдімо поріг, який її мінімізує.


In [ ]:
print("ціна пропуску   поріг  precision  recall    втрата")
for price in (0.2, 0.5, 1, 2, 5, 10, 20):
    losses = [c[1] + price * c[2] for c in curve]
    best = min(range(101), key=lambda i: losses[i])
    print("%8.1f × FP   %.2f   %.4f    %.4f   %8.0f"
          % (price, grid[best], curve[best][4], curve[best][5], losses[best]))


## 11 · Три зерна: яка різниця є різницею

Одне число нічого не варте без розкиду навколо нього. Правило курсу просте:
**різниця, менша за розкид, не є різницею**. Проженемо чотири моделі через три
поділи корпусу.


In [ ]:
MODELS = {
    "LogReg":           lambda seed: LogisticRegression(max_iter=1000, random_state=seed),
    "LogReg balanced":  lambda seed: LogisticRegression(max_iter=1000, random_state=seed,
                                                        class_weight='balanced'),
    "ComplementNB":     lambda seed: ComplementNB(),
    "MultinomialNB":    lambda seed: MultinomialNB(),
}
seed_results = {name: {"f1_pos": [], "macro": [], "accuracy": []} for name in MODELS}

for seed in (0, 1, 2):
    A, B, y_tr, y_te = split_and_vectorize(seed)
    for name, make in MODELS.items():
        p = make(seed).fit(A, y_tr).predict(B)
        seed_results[name]["f1_pos"].append(f1_score(y_te, p, pos_label=1))
        seed_results[name]["macro"].append(f1_score(y_te, p, average='macro'))
        seed_results[name]["accuracy"].append(accuracy_score(y_te, p))

print("модель             F1 «помилка»      макро-F1         частка правильних")
for name, r in seed_results.items():
    print("%-17s %.4f ±%.4f  %.4f ±%.4f  %.4f ±%.4f"
          % (name,
             np.mean(r["f1_pos"]), np.std(r["f1_pos"]),
             np.mean(r["macro"]), np.std(r["macro"]),
             np.mean(r["accuracy"]), np.std(r["accuracy"])))


### Дві моделі, три метрики, три різні вердикти

Найкорисніше порівняння тут — `MultinomialNB` проти `ComplementNB`. Купа зерен —
це відрізок від найгіршого зерна до найкращого. Якщо купи двох моделей
**перетинаються**, різниці не доведено.


In [ ]:
def pile(name, metric):
    values = seed_results[name][metric]
    return min(values), max(values), float(np.mean(values))

def compare(left, right, metric):
    lo1, hi1, m1 = pile(left, metric)
    lo2, hi2, m2 = pile(right, metric)
    overlap = not (hi1 < lo2 or hi2 < lo1)
    print("%-18s %s [%.4f … %.4f]   %s [%.4f … %.4f]   %s"
          % (metric, left, lo1, hi1, right, lo2, hi2,
             "КУПИ ПЕРЕТИНАЮТЬСЯ — різниці не доведено" if overlap
             else "купи не перетинаються, різниця %+.4f" % (m2 - m1)))

for metric in ("accuracy", "f1_pos", "macro"):
    compare("MultinomialNB", "ComplementNB", metric)
print()
for metric in ("accuracy", "f1_pos", "macro"):
    compare("LogReg", "LogReg balanced", metric)


## 12 · Підсумок теми в одній таблиці


In [ ]:
summary = [
    ("частка класу «помилка»",                 labels.mean()),
    ("F1 класу «помилка», LogReg",             F1_pos),
    ("F1 класу «не помилка», LogReg",          F1_neg),
    ("частка правильних, LogReg",              accuracy),
    ("мікро-F1, LogReg",                       micro),
    ("макро-F1, LogReg",                       macro),
    ("зважена F1, LogReg",                     weighted),
    ("частка правильних, дурна модель",        dumb_accuracy),
    ("макро-F1, дурна модель",                 dumb_macro),
]
for name, value in summary:
    print("%-34s %.4f" % (name, value))

print()
print("зошит виконався за %.0f секунд процесорного часу (%.0f стінних)"
      % (time.process_time() - NOTEBOOK_CPU, time.time() - NOTEBOOK_START))


## Завдання

### 🟢 Рівень 1 — База

Візьми поріг `0.27` (той, що дає найбільшу F1) і надрукуй **повну** матрицю
плутанини разом із усіма чотирма метриками класу «помилка». Порівняй із порогом
`0.5` і скажи словами, що саме змінилось: скільки хибних тривог додалося і
скільки пропусків зникло.

### 🟡 Рівень 2 — Плюс

Побудуй свою «дурну» модель для **багатокласової** задачі: нехай вона відповідає
випадковою програмою, обраною пропорційно до розміру класу. Порахуй її мікро-F1
і макро-F1 на трьох зернах. Порівняй із моделлю «усе — найбільша програма» і
поясни, чому в них різні мікро й майже однакові макро.

### 🔴 Рівень 3 — Виклик

Напиши власну функцію `micro_f1(y_true, y_pred)` без жодного виклику `sklearn` і
доведи `assert`-ом, що вона дорівнює `accuracy_score` на **трьох** різних наборах:
двокласовому, багатокласовому й на випадкових мітках. Потім зламай рівність:
зроби набір, де приклад дістає **дві** відповіді, і покажи, що там мікро-F1 уже не
дорівнює частці правильних.
